# OpenStream — BERTweet Guard v6

**BERTweet + LoRA (r=32) + Four-Source Moderation Corpus + Source/Class-Weighted Sampling + Cross-Entropy + Cosine Warmup + Validation-Tuned Threshold**

v6 is intentionally a clean rebuild after the v5 experiments.

Training data:
1. **Jigsaw Toxic Comment Classification** — Wikipedia comments; adds toxicity, threat, obscenity, insult, identity-hate coverage.
2. **HateXplain** — Twitter/Gab social-media posts.
3. **Davidson Hate Speech & Offensive Language** — Twitter.
4. **TweetEval Offensive** — Twitter offensive/non-offensive benchmark.

All available labeled rows from those sources are kept in the data pool. Each source is split independently **90 / 5 / 5**. The training loader uses a weighted sampler so large Jigsaw and class imbalance do not dominate learning; no source is destructively downsampled.

**OLID/OffensEval is not added separately** because TweetEval Offensive is based on that benchmark and adding both would duplicate the same underlying task data.

The serving contract is unchanged: `normal=0`, `flagged=1`, threshold-based classification, and the same response dictionary used by the existing applications.

A fixed 15-case regression suite runs on the final disk artifact **before packaging/downloading**.


## 1. Environment

In [1]:
# Colab can ship an old torchao build that newer PEFT detects and rejects.
# BERTweet Guard v6 does not use torchao, so remove it before importing PEFT.
!pip uninstall -y torchao >/dev/null 2>&1

# Keep Colab's existing PyTorch/CUDA build; install only the libraries we need.
!pip install -q -U transformers peft scikit-learn pandas matplotlib seaborn emoji datasets huggingface_hub

print("Environment ready: optional torchao removed; Colab PyTorch left unchanged.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 127.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 142.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 141.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 137.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 610.9/610.9 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.9/842.9 kB 62.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requir

**v6 environment note:** Colab may come with an old optional `torchao` package. Newer PEFT versions can reject that package even though this project does not use it. The setup cell removes `torchao` and keeps Colab's existing PyTorch/CUDA installation unchanged.


In [2]:
import importlib.util
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"torchao installed: {importlib.util.find_spec('torchao') is not None}")

if importlib.util.find_spec('torchao') is not None:
    raise RuntimeError(
        "torchao is still installed. Restart the runtime once, then Run all from the top."
    )


PyTorch: 2.11.0+cu128
CUDA available: True
torchao installed: False


In [3]:
import os, re, json, warnings, shutil
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

from transformers import (
    AutoTokenizer,
    BertweetTokenizer,
    AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup,
    logging as hf_logging,
)
hf_logging.set_verbosity_error()

from peft import get_peft_model, LoraConfig, TaskType, PeftModel
from datasets import load_dataset
from huggingface_hub import hf_hub_download

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns


## 2. Configuration

In [4]:
RANDOM_SEED   = 42
MODEL_NAME    = "vinai/bertweet-base"

MAX_LENGTH    = 128
BATCH_SIZE    = 32
NUM_LABELS    = 2

# Keep the architecture stable so v6 primarily tests the data/sampling strategy.
LORA_R        = 32
LORA_ALPHA    = 64
LORA_DROPOUT  = 0.20
LORA_TARGETS  = ["query", "key", "value", "dense"]

LEARNING_RATE = 5e-5
WEIGHT_DECAY  = 0.05
NUM_EPOCHS    = 5
PATIENCE      = 2
WARMUP_RATIO  = 0.06

RAW_DIR       = "data/raw_v6"
PROCESSED_DIR = "data/processed_v6_four_source"
OUT_DIR       = "outputs/bertweet_lora32_four_source_v6"
ARTIFACT_DIR  = "artifacts/openstream-moderation-v6"
ARCHIVE_NAME  = "openstream-moderation-model-v6"

LABEL2ID = {"normal": 0, "flagged": 1}
ID2LABEL = {0: "normal", 1: "flagged"}

for d in [RAW_DIR, PROCESSED_DIR, OUT_DIR]:
    os.makedirs(d, exist_ok=True)

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")


Device : cuda
PyTorch: 2.11.0+cu128


## 3. Four-source moderation corpus

v6 combines four unique sources:

- **Jigsaw** for broad toxicity/threat/insult/obscenity coverage.
- **HateXplain** for social-media hate/offensive/normal language.
- **Davidson** for Twitter hate/offensive/neither language.
- **TweetEval Offensive** for Twitter offensive/non-offensive language.

All sources are mapped to the unchanged binary contract:

- `normal = 0`
- `flagged = 1`

TweetEval Offensive already represents the OLID/OffensEval task, so OLID is not added again as a separate fourth/fifth copy.


In [5]:
# Dataset identifiers used in v6
JIGSAW_DATASET   = "thesofakillers/jigsaw-toxic-comment-classification-challenge"
HATEXPLAIN_DATASET = "literAlbDev/hatexplain"
DAVIDSON_DATASET = "tdavidson/hate_speech_offensive"
TWEETEVAL_DATASET = "cardiffnlp/tweet_eval"


In [6]:
SOURCE_JIGSAW = "jigsaw"
SOURCE_HATEXPLAIN = "hatexplain"
SOURCE_DAVIDSON = "davidson"
SOURCE_TWEETEVAL = "tweet_eval_offensive"

JIGSAW_FLAG_COLUMNS = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate",
]


def _clean_basic(text):
    if not isinstance(text, str):
        return ""
    return re.sub(r"\s+", " ", text).strip()


def load_jigsaw_frame():
    """
    Original Jigsaw train split.
    Binary mapping:
      none of the six toxicity flags -> normal
      any toxicity flag == 1        -> flagged
    """
    ds = load_dataset(JIGSAW_DATASET, split="train")
    frame = ds.to_pandas()

    missing = [c for c in ["comment_text", *JIGSAW_FLAG_COLUMNS] if c not in frame.columns]
    if missing:
        raise ValueError(f"Jigsaw is missing expected columns: {missing}")

    flag_matrix = frame[JIGSAW_FLAG_COLUMNS].fillna(0).astype(float)
    labels = (flag_matrix.max(axis=1) > 0).astype(int)

    return pd.DataFrame({
        "text": frame["comment_text"].map(_clean_basic),
        "label": labels,
        "source": SOURCE_JIGSAW,
    })


def load_hatexplain_frame():
    """
    HateXplain annotation ids:
      0 = hatespeech
      1 = normal
      2 = offensive

    Require a real majority (>= 2 annotators). Fully ambiguous rows are dropped.
    """
    ds = load_dataset(HATEXPLAIN_DATASET)
    rows = []

    for _, split_ds in ds.items():
        for row in split_ds:
            annotations = row.get("annotators", {})

            if isinstance(annotations, dict):
                raw_labels = annotations.get("label", [])
            else:
                raw_labels = [
                    a.get("label")
                    for a in annotations
                    if isinstance(a, dict) and "label" in a
                ]

            labels = []
            for value in raw_labels:
                if isinstance(value, str):
                    value = {
                        "hatespeech": 0,
                        "hate": 0,
                        "normal": 1,
                        "offensive": 2,
                    }.get(value.lower(), value)

                try:
                    labels.append(int(value))
                except Exception:
                    pass

            if not labels:
                continue

            counts = Counter(labels)
            majority_label, majority_count = counts.most_common(1)[0]

            if majority_count < 2:
                continue

            text = " ".join(row.get("post_tokens", []))
            binary_label = (
                LABEL2ID["normal"]
                if majority_label == 1
                else LABEL2ID["flagged"]
            )

            rows.append({
                "text": _clean_basic(text),
                "label": binary_label,
                "source": SOURCE_HATEXPLAIN,
            })

    return pd.DataFrame(rows)


def load_davidson_frame():
    """
    Davidson:
      class 0 = hate speech
      class 1 = offensive language
      class 2 = neither
    """
    ds = load_dataset(DAVIDSON_DATASET, split="train")
    frame = ds.to_pandas()

    class_col = "class" if "class" in frame.columns else "label"
    labels = (frame[class_col].astype(int) != 2).astype(int)

    return pd.DataFrame({
        "text": frame["tweet"].map(_clean_basic),
        "label": labels,
        "source": SOURCE_DAVIDSON,
    })


def load_tweeteval_offensive_frame():
    """
    TweetEval Offensive:
      0 = non-offensive
      1 = offensive

    Combine its published splits, then perform the same v6 source-aware 90/5/5
    split as every other source.
    """
    ds = load_dataset(TWEETEVAL_DATASET, "offensive")
    parts = []

    for _, split_ds in ds.items():
        frame = split_ds.to_pandas()[["text", "label"]].copy()
        frame["text"] = frame["text"].map(_clean_basic)
        frame["label"] = frame["label"].astype(int)
        frame["source"] = SOURCE_TWEETEVAL
        parts.append(frame)

    return pd.concat(parts, ignore_index=True)


print("Loading Jigsaw...")
jg_df = load_jigsaw_frame()

print("Loading HateXplain...")
hx_df = load_hatexplain_frame()

print("Loading Davidson...")
dv_df = load_davidson_frame()

print("Loading TweetEval Offensive...")
te_df = load_tweeteval_offensive_frame()

df = pd.concat(
    [jg_df, hx_df, dv_df, te_df],
    ignore_index=True,
)

df = df[df["text"].str.len() > 0].copy()

print("\nRaw harmonized source/class counts:")
print(df.groupby(["source", "label"]).size().unstack(fill_value=0))
print(f"\nRows before de-duplication: {len(df):,}")


Loading Jigsaw...


README.md:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

train.csv: reconstructing file:   0%|          |  0.00B / 68.8MB            

train.csv: downloading bytes:           |  0.00B            

test.csv: reconstructing file:   0%|          |  0.00B / 60.4MB            

...

offensive/validation-00000-of-00001.parq(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/11916 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/860 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1324 [00:00<?, ? examples/s]


Raw harmonized source/class counts:
label                      0      1
source                             
davidson                4163  20620
hatexplain              7814  11415
jigsaw                143346  16225
tweet_eval_offensive    9460   4640

Rows before de-duplication: 217,683


### 3.1 Cross-source de-duplication

De-duplication happens **before** splitting. If the same normalized text has conflicting binary labels, all copies of that text are dropped rather than guessing. Same-label duplicates are kept once.


In [7]:
df["dedup_key"] = (
    df["text"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

label_counts_per_text = df.groupby("dedup_key")["label"].nunique()
conflicting_keys = set(
    label_counts_per_text[label_counts_per_text > 1].index
)

if conflicting_keys:
    print(f"Dropping {len(conflicting_keys):,} text keys with conflicting labels.")
    df = df[~df["dedup_key"].isin(conflicting_keys)].copy()

before_same_label_dedup = len(df)
df = df.drop_duplicates(subset=["dedup_key"], keep="first").copy()
same_label_duplicates_removed = before_same_label_dedup - len(df)

df = df.drop(columns=["dedup_key"]).reset_index(drop=True)

print(f"Same-label duplicates removed: {same_label_duplicates_removed:,}")
print(f"Rows after de-duplication : {len(df):,}")
print("\nFinal source/class counts:")
print(df.groupby(["source", "label"]).size().unstack(fill_value=0))


Dropping 29 text keys with conflicting labels.
Same-label duplicates removed: 336
Rows after de-duplication : 217,270

Final source/class counts:
label                      0      1
source                             
davidson                4163  20607
hatexplain              7787  11398
jigsaw                143131  16169
tweet_eval_offensive    9399   4616


### 3.2 BERTweet-aligned preprocessing

Only normalize whitespace here. BERTweet-specific normalization stays inside the tokenizer via `normalization=True`. Do not globally lowercase, strip hashtags, or replace mentions/URLs with custom tokens.


In [8]:
print("Preparing four-source text for BERTweet...")

df["text"] = (
    df["text"]
    .fillna("")
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df = df[df["text"].str.len() > 0].copy()
df["label"] = df["label"].astype(int)
df = df[["text", "label", "source"]].reset_index(drop=True)

print(f"Usable rows: {len(df):,}")


Preparing four-source text for BERTweet...
Usable rows: 217,270


In [9]:
def class_report(frame: pd.DataFrame, split_name: str = "full") -> None:
    counts = Counter(frame["label"].tolist())
    total = len(frame)

    print(f"[{split_name}] n={total:,}")

    for lid in sorted(ID2LABEL):
        n = counts.get(lid, 0)
        pct = 100 * n / total if total else 0
        print(f"{ID2LABEL[lid]:>10} ({lid})  {n:7,}  {pct:5.1f}%")

    if "source" in frame.columns:
        print("  by source:")
        table = (
            frame.groupby(["source", "label"])
            .size()
            .unstack(fill_value=0)
            .rename(columns=ID2LABEL)
        )
        print(table)


class_report(df, "full")


[full] n=217,270
    normal (0)  164,480   75.7%
   flagged (1)   52,790   24.3%
  by source:
label                 normal  flagged
source                               
davidson                4163    20607
hatexplain              7787    11398
jigsaw                143131    16169
tweet_eval_offensive    9399     4616


### 3.3 Source-aware split — 90 / 5 / 5

Every source is independently stratified 90/5/5 and then the corresponding pieces are combined.

**No destructive class downsampling is performed in v6.** Every training row remains available to the sampler. Validation and test keep their natural source/class distributions.


In [10]:
def split_source_90_5_5(source_df: pd.DataFrame):
    source_df = source_df.reset_index(drop=True)

    train_part, temp_part = train_test_split(
        source_df,
        test_size=0.10,
        stratify=source_df["label"],
        random_state=RANDOM_SEED,
    )

    val_part, test_part = train_test_split(
        temp_part,
        test_size=0.50,
        stratify=temp_part["label"],
        random_state=RANDOM_SEED,
    )

    return train_part, val_part, test_part


train_parts, val_parts, test_parts = [], [], []

for source_name, source_df in df.groupby("source"):
    tr, va, te = split_source_90_5_5(source_df)
    train_parts.append(tr)
    val_parts.append(va)
    test_parts.append(te)

train_df = (
    pd.concat(train_parts, ignore_index=True)
    .sample(frac=1.0, random_state=RANDOM_SEED)
    .reset_index(drop=True)
)

val_df = (
    pd.concat(val_parts, ignore_index=True)
    .sample(frac=1.0, random_state=RANDOM_SEED)
    .reset_index(drop=True)
)

test_df = (
    pd.concat(test_parts, ignore_index=True)
    .sample(frac=1.0, random_state=RANDOM_SEED)
    .reset_index(drop=True)
)

for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
    split.to_csv(
        os.path.join(PROCESSED_DIR, f"{name}.csv"),
        index=False,
    )
    print()
    class_report(split, name)



[train] n=195,542
    normal (0)  148,032   75.7%
   flagged (1)   47,510   24.3%
  by source:
label                 normal  flagged
source                               
davidson                3747    18546
hatexplain              7008    10258
jigsaw                128818    14552
tweet_eval_offensive    8459     4154

[val] n=10,863
    normal (0)    8,223   75.7%
   flagged (1)    2,640   24.3%
  by source:
label                 normal  flagged
source                               
davidson                 208     1030
hatexplain               389      570
jigsaw                  7156      809
tweet_eval_offensive     470      231

[test] n=10,865
    normal (0)    8,225   75.7%
   flagged (1)    2,640   24.3%
  by source:
label                 normal  flagged
source                               
davidson                 208     1031
hatexplain               390      570
jigsaw                  7157      808
tweet_eval_offensive     470      231


### 3.4 Source/class-weighted training sampler

Jigsaw is much larger than the other sources and several datasets have strong class imbalance. Instead of deleting examples, v6 assigns each training row a sampling weight inversely proportional to its `(source, label)` group size.

This gives each source/class group comparable expected exposure while keeping **all rows** in the training pool.


In [11]:
group_counts = (
    train_df.groupby(["source", "label"])
    .size()
    .to_dict()
)

train_sample_weights = np.asarray(
    [
        1.0 / group_counts[(source, int(label))]
        for source, label in zip(
            train_df["source"],
            train_df["label"],
        )
    ],
    dtype=np.float64,
)

print("Training source/class group sizes:")
for key, value in sorted(group_counts.items()):
    print(f"  {key}: {value:,}")

print(
    f"\nSampler pool: {len(train_df):,} unique training rows; "
    f"{len(train_df):,} weighted draws per epoch."
)


Training source/class group sizes:
  ('davidson', 0): 3,747
  ('davidson', 1): 18,546
  ('hatexplain', 0): 7,008
  ('hatexplain', 1): 10,258
  ('jigsaw', 0): 128,818
  ('jigsaw', 1): 14,552
  ('tweet_eval_offensive', 0): 8,459
  ('tweet_eval_offensive', 1): 4,154

Sampler pool: 195,542 unique training rows; 195,542 weighted draws per epoch.


## 4. Dataset, BERTweet tokenizer, and weighted loaders


In [12]:
class TweetDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, tokenizer):
        self.texts = frame["text"].tolist()
        self.labels = frame["label"].tolist()
        self.tok = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tok(
            self.texts[idx],
            max_length=MAX_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(
                self.labels[idx],
                dtype=torch.long,
            ),
        }


def make_loaders(tokenizer):
    frames = {}

    for name in ["train", "val", "test"]:
        frame = pd.read_csv(
            os.path.join(PROCESSED_DIR, f"{name}.csv")
        )

        frame["text"] = frame["text"].fillna("").astype(str)
        frame["label"] = frame["label"].astype(int)

        frames[name] = frame.reset_index(drop=True)

    train_counts = (
        frames["train"]
        .groupby(["source", "label"])
        .size()
        .to_dict()
    )

    weights = torch.as_tensor(
        [
            1.0 / train_counts[(source, int(label))]
            for source, label in zip(
                frames["train"]["source"],
                frames["train"]["label"],
            )
        ],
        dtype=torch.double,
    )

    sampler_generator = torch.Generator()
    sampler_generator.manual_seed(RANDOM_SEED)

    train_sampler = WeightedRandomSampler(
        weights=weights,
        num_samples=len(frames["train"]),
        replacement=True,
        generator=sampler_generator,
    )

    common_kw = dict(
        batch_size=BATCH_SIZE,
        num_workers=2,
        pin_memory=(DEVICE.type == "cuda"),
    )

    train_loader = DataLoader(
        TweetDataset(frames["train"], tokenizer),
        sampler=train_sampler,
        shuffle=False,
        **common_kw,
    )

    val_loader = DataLoader(
        TweetDataset(frames["val"], tokenizer),
        shuffle=False,
        **common_kw,
    )

    test_loader = DataLoader(
        TweetDataset(frames["test"], tokenizer),
        shuffle=False,
        **common_kw,
    )

    return (
        train_loader,
        val_loader,
        test_loader,
        frames["train"],
        frames["val"],
        frames["test"],
    )


In [13]:
print("Loading original BERTweet tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False,
    normalization=True,
)

diag_frame = train_df.sample(
    n=min(20000, len(train_df)),
    random_state=RANDOM_SEED,
).copy()

diag_frame["token_length"] = [
    len(
        tokenizer.encode(
            text,
            add_special_tokens=True,
            truncation=False,
        )
    )
    for text in diag_frame["text"].tolist()
]

print("\nOverall token-length diagnostic:")
print(f"  mean             : {diag_frame['token_length'].mean():.1f}")
print(f"  median           : {diag_frame['token_length'].median():.1f}")
print(f"  95th percentile  : {diag_frame['token_length'].quantile(0.95):.1f}")
print(
    f"  > {MAX_LENGTH} tokens : "
    f"{(diag_frame['token_length'] > MAX_LENGTH).mean():.2%}"
)

print("\nToken length by source and class:")
print(
    diag_frame.groupby(["source", "label"])["token_length"]
    .agg(["count", "mean", "median", "max"])
    .round(1)
)

(
    train_loader,
    val_loader,
    test_loader,
    train_df_raw,
    val_df_raw,
    test_df_raw,
) = make_loaders(tokenizer)

print(
    f"\nBatches — train: {len(train_loader):,}  "
    f"val: {len(val_loader):,}  "
    f"test: {len(test_loader):,}"
)


Loading original BERTweet tokenizer...


config.json:   0%|          | 0.00/558 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/843k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.91M [00:00<?, ?B/s]


Overall token-length diagnostic:
  mean             : 72.8
  median           : 36.0
  95th percentile  : 243.1
  > 128 tokens : 13.11%

Token length by source and class:
                            count  mean  median   max
source               label                           
davidson             0        354  21.6    22.0    53
                     1       1900  19.3    18.0    48
hatexplain           0        690  29.0    25.0    79
                     1       1036  29.0    26.0   151
jigsaw               0      13216  91.7    51.0  1420
                     1       1481  75.2    33.0  2502
tweet_eval_offensive 0        881  28.1    23.0   104
                     1        442  30.8    27.0    90

Batches — train: 6,111  val: 340  test: 340


## 5. Cross-entropy loss

v6 uses standard cross-entropy with **no extra class weights**. Source/class balancing is handled by the weighted sampler rather than by deleting data or applying an additional flagged-class boost.


In [14]:
loss_fn = nn.CrossEntropyLoss()
print("CrossEntropyLoss ready (no class weights)")


CrossEntropyLoss ready (no class weights)


## 6. Train and evaluation loops

Checkpoint and threshold selection use the **mean Macro F1 across sources** on validation data, so the much larger Jigsaw validation partition cannot dominate the operating threshold.


In [15]:
def run_train_epoch(model, loader, optimizer, scheduler, loss_fn):
    model.train()
    total = 0.0

    for batch in loader:
        optimizer.zero_grad()

        logits = model(
            input_ids=batch["input_ids"].to(DEVICE),
            attention_mask=batch["attention_mask"].to(DEVICE),
        ).logits

        loss = loss_fn(
            logits,
            batch["labels"].to(DEVICE),
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0,
        )

        optimizer.step()
        scheduler.step()

        total += loss.item()

    return total / len(loader)


def run_eval_probs(model, loader, loss_fn):
    model.eval()

    total = 0.0
    probs_all = []
    labs_all = []

    with torch.no_grad():
        for batch in loader:
            labels = batch["labels"].to(DEVICE)

            logits = model(
                input_ids=batch["input_ids"].to(DEVICE),
                attention_mask=batch["attention_mask"].to(DEVICE),
            ).logits

            total += loss_fn(logits, labels).item()

            probs = torch.softmax(
                logits,
                dim=-1,
            )[:, 1]

            probs_all.extend(
                probs.detach().cpu().numpy()
            )

            labs_all.extend(
                labels.detach().cpu().numpy()
            )

    return (
        total / len(loader),
        np.asarray(probs_all),
        np.asarray(labs_all),
    )


def threshold_scores(
    probs,
    labels,
    sources,
    threshold,
):
    preds = (probs >= threshold).astype(int)

    overall_macro_f1 = f1_score(
        labels,
        preds,
        average="macro",
        zero_division=0,
    )

    source_f1 = {}

    for source_name in sorted(np.unique(sources)):
        mask = sources == source_name

        source_f1[source_name] = f1_score(
            labels[mask],
            preds[mask],
            average="macro",
            zero_division=0,
        )

    source_balanced_macro_f1 = float(
        np.mean(list(source_f1.values()))
    )

    return (
        source_balanced_macro_f1,
        float(overall_macro_f1),
        source_f1,
    )


def find_best_threshold(
    probs,
    labels,
    sources,
    start=0.20,
    stop=0.80,
    step=0.005,
):
    thresholds = np.arange(
        start,
        stop + step / 2,
        step,
    )

    best_threshold = 0.50
    best_source_balanced_f1 = -1.0
    best_overall_f1 = -1.0

    for threshold in thresholds:
        source_balanced_f1, overall_f1, _ = threshold_scores(
            probs,
            labels,
            sources,
            threshold,
        )

        improved = (
            source_balanced_f1 > best_source_balanced_f1
            or (
                np.isclose(
                    source_balanced_f1,
                    best_source_balanced_f1,
                )
                and overall_f1 > best_overall_f1
            )
        )

        if improved:
            best_threshold = float(threshold)
            best_source_balanced_f1 = float(
                source_balanced_f1
            )
            best_overall_f1 = float(overall_f1)

    return (
        best_threshold,
        best_source_balanced_f1,
        best_overall_f1,
    )


In [16]:
def save_plot_history(history: dict, title: str, save_path: str):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history["train_loss"], label="Train")
    axes[0].plot(history["val_loss"], label="Val")
    axes[0].set_title(f"{title} — Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(
        history["val_source_balanced_f1"],
        label="Source-balanced Val Macro F1",
    )
    axes[1].plot(
        history["val_overall_macro_f1"],
        label="Overall Val Macro F1",
    )
    axes[1].set_title(f"{title} — Validation F1")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    plt.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)

    print(f"Saved history plot: {save_path}")


def save_cm(
    labels,
    preds,
    title: str,
    save_path: str,
    cmap: str = "Greens",
):
    cm = confusion_matrix(labels, preds)

    fig, ax = plt.subplots(figsize=(6, 5))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap=cmap,
        xticklabels=list(ID2LABEL.values()),
        yticklabels=list(ID2LABEL.values()),
        ax=ax,
    )

    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)

    plt.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)

    print(f"Saved confusion matrix: {save_path}")


In [17]:
def training_loop_cosine(
    model,
    train_loader,
    val_loader,
    val_sources,
    loss_fn,
    out_dir: str,
    lr: float,
    label: str,
    num_epochs: int = NUM_EPOCHS,
    patience: int = PATIENCE,
):
    os.makedirs(out_dir, exist_ok=True)

    optimizer = AdamW(
        filter(
            lambda p: p.requires_grad,
            model.parameters(),
        ),
        lr=lr,
        weight_decay=WEIGHT_DECAY,
    )

    total_steps = len(train_loader) * num_epochs
    warmup_steps = int(
        WARMUP_RATIO * total_steps
    )

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    best_source_balanced_f1 = -1.0
    best_overall_f1 = -1.0
    best_val_threshold = 0.50

    best_ckpt = os.path.join(
        out_dir,
        "best_model",
    )

    patience_ctr = 0

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_source_balanced_f1": [],
        "val_overall_macro_f1": [],
        "val_threshold": [],
    }

    for epoch in range(1, num_epochs + 1):
        train_loss = run_train_epoch(
            model,
            train_loader,
            optimizer,
            scheduler,
            loss_fn,
        )

        val_loss, val_probs, val_labels = run_eval_probs(
            model,
            val_loader,
            loss_fn,
        )

        (
            val_threshold,
            val_source_balanced_f1,
            val_overall_f1,
        ) = find_best_threshold(
            val_probs,
            val_labels,
            val_sources,
            start=0.20,
            stop=0.80,
            step=0.005,
        )

        history["train_loss"].append(
            float(train_loss)
        )
        history["val_loss"].append(
            float(val_loss)
        )
        history["val_source_balanced_f1"].append(
            float(val_source_balanced_f1)
        )
        history["val_overall_macro_f1"].append(
            float(val_overall_f1)
        )
        history["val_threshold"].append(
            float(val_threshold)
        )

        improved = (
            val_source_balanced_f1
            > best_source_balanced_f1
        )

        marker = "  <- best" if improved else ""

        print(
            f"[{label}] Epoch {epoch:2d}/{num_epochs}  "
            f"train_loss={train_loss:.4f}  "
            f"val_loss={val_loss:.4f}  "
            f"val_source_f1={val_source_balanced_f1:.4f}  "
            f"val_macro_f1={val_overall_f1:.4f}  "
            f"threshold={val_threshold:.3f}"
            f"{marker}"
        )

        if improved:
            best_source_balanced_f1 = float(
                val_source_balanced_f1
            )
            best_overall_f1 = float(
                val_overall_f1
            )
            best_val_threshold = float(
                val_threshold
            )

            patience_ctr = 0

            if os.path.exists(best_ckpt):
                shutil.rmtree(best_ckpt)

            os.makedirs(
                best_ckpt,
                exist_ok=True,
            )

            # Save adapter/model state only. Do NOT save a BERTweet tokenizer
            # into the checkpoint; v6 packages the original tokenizer files
            # explicitly during final export.
            model.save_pretrained(best_ckpt)

        else:
            patience_ctr += 1

            if patience_ctr >= patience:
                print(
                    f"[{label}] Early stopping at epoch "
                    f"{epoch} (patience={patience})"
                )
                break

    save_plot_history(
        history,
        label,
        os.path.join(
            out_dir,
            "training_history.png",
        ),
    )

    print(
        f"\n[{label}] Best source-balanced Val Macro F1 = "
        f"{best_source_balanced_f1:.4f}"
    )

    print(
        f"[{label}] Overall Val Macro F1 at best epoch = "
        f"{best_overall_f1:.4f}"
    )

    print(
        f"[{label}] Best-epoch validation threshold = "
        f"{best_val_threshold:.3f}"
    )

    print(
        f"[{label}] Warmup steps = "
        f"{warmup_steps:,} / {total_steps:,}"
    )

    with open(
        os.path.join(out_dir, "history.json"),
        "w",
    ) as f:
        json.dump(
            history,
            f,
            indent=2,
        )

    return (
        best_ckpt,
        best_source_balanced_f1,
        best_overall_f1,
        best_val_threshold,
    )


## 7. Model

The LoRA architecture remains the same as v5 so the main experiment change is the four-source corpus, weighted sampling, and cross-entropy objective.


In [18]:
def build_model():
    base = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )

    cfg = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGETS,
        bias="none",
    )

    model = get_peft_model(
        base,
        cfg,
    )

    model.print_trainable_parameters()

    return model


model = build_model().to(DEVICE)


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  543MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  543MB            

model.safetensors: downloading bytes:           |  0.00B            

trainable params: 5,900,546 || all params: 140,802,052 || trainable%: 4.1907


## 8. Train

The best checkpoint is selected using **source-balanced validation Macro F1**, while the threshold is also tuned on validation only.


In [19]:
LABEL = "BERTweet + LoRA r=32 + Four-Source Weighted Sampling + CrossEntropy"

val_sources = val_df_raw["source"].to_numpy()

(
    best_ckpt,
    best_val_source_f1,
    best_val_overall_f1,
    best_epoch_threshold,
) = training_loop_cosine(
    model,
    train_loader,
    val_loader,
    val_sources,
    loss_fn,
    OUT_DIR,
    LEARNING_RATE,
    LABEL,
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE,
)


[BERTweet + LoRA r=32 + Four-Source Weighted Sampling + CrossEntropy] Epoch  1/5  train_loss=0.3394  val_loss=0.2480  val_source_f1=0.8272  val_macro_f1=0.8902  threshold=0.735  <- best
[BERTweet + LoRA r=32 + Four-Source Weighted Sampling + CrossEntropy] Epoch  2/5  train_loss=0.2401  val_loss=0.2265  val_source_f1=0.8362  val_macro_f1=0.9052  threshold=0.800  <- best
[BERTweet + LoRA r=32 + Four-Source Weighted Sampling + CrossEntropy] Epoch  3/5  train_loss=0.1903  val_loss=0.2519  val_source_f1=0.8298  val_macro_f1=0.8963  threshold=0.780
[BERTweet + LoRA r=32 + Four-Source Weighted Sampling + CrossEntropy] Epoch  4/5  train_loss=0.1584  val_loss=0.2432  val_source_f1=0.8381  val_macro_f1=0.9069  threshold=0.800  <- best
[BERTweet + LoRA r=32 + Four-Source Weighted Sampling + CrossEntropy] Epoch  5/5  train_loss=0.1433  val_loss=0.2656  val_source_f1=0.8362  val_macro_f1=0.9020  threshold=0.750
Saved history plot: outputs/bertweet_lora32_four_source_v6/training_history.png

[BERTwe

---

## 9. Final validation threshold and held-out test evaluation

The best checkpoint is reloaded. The final threshold is refined at 0.001 resolution using **validation data only**, optimizing the mean Macro F1 across the four sources. Test data remains untouched until the final evaluation.


In [20]:
base_for_eval = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

model_eval = PeftModel.from_pretrained(
    base_for_eval,
    best_ckpt,
).to(DEVICE)

model_eval.eval()

val_loss_final, val_probs, val_labels = run_eval_probs(
    model_eval,
    val_loader,
    loss_fn,
)

val_sources = val_df_raw["source"].to_numpy()

(
    best_threshold,
    best_threshold_source_f1,
    best_threshold_overall_f1,
) = find_best_threshold(
    val_probs,
    val_labels,
    val_sources,
    start=0.20,
    stop=0.80,
    step=0.001,
)

print(
    f"Best validation threshold              : "
    f"{best_threshold:.3f}"
)

print(
    f"Source-balanced Val Macro F1           : "
    f"{best_threshold_source_f1:.4f}"
)

print(
    f"Overall Val Macro F1                   : "
    f"{best_threshold_overall_f1:.4f}"
)

with open(
    os.path.join(OUT_DIR, "threshold.json"),
    "w",
) as f:
    json.dump(
        {
            "threshold": best_threshold,
            "validation_source_balanced_macro_f1":
                best_threshold_source_f1,
            "validation_overall_macro_f1":
                best_threshold_overall_f1,
            "selection":
                "validation source-balanced Macro F1 sweep "
                "from 0.20 to 0.80 in 0.001 increments",
        },
        f,
        indent=2,
    )

# Test is evaluated only after checkpoint + threshold are fixed.
_, test_probs, test_labels = run_eval_probs(
    model_eval,
    test_loader,
    loss_fn,
)

test_preds = (
    test_probs >= best_threshold
).astype(int)

test_f1 = f1_score(
    test_labels,
    test_preds,
    average="macro",
    zero_division=0,
)

test_accuracy = accuracy_score(
    test_labels,
    test_preds,
)

test_sources = test_df_raw["source"].to_numpy()

(
    test_source_balanced_f1,
    test_overall_f1_check,
    test_source_f1,
) = threshold_scores(
    test_probs,
    test_labels,
    test_sources,
    best_threshold,
)

report = classification_report(
    test_labels,
    test_preds,
    target_names=list(ID2LABEL.values()),
    output_dict=True,
    zero_division=0,
)

print(
    classification_report(
        test_labels,
        test_preds,
        target_names=list(ID2LABEL.values()),
        zero_division=0,
    )
)

print(f"Selected Threshold             : {best_threshold:.3f}")
print(f"Test Overall Macro F1          : {test_f1:.4f}")
print(f"Test Source-Balanced Macro F1  : {test_source_balanced_f1:.4f}")
print(f"Test Accuracy                  : {test_accuracy:.4f}")

print("\nPer-source Test Macro F1:")
for source_name, score in test_source_f1.items():
    print(f"  {source_name:>24}: {score:.4f}")


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Best validation threshold              : 0.799
Source-balanced Val Macro F1           : 0.8381
Overall Val Macro F1                   : 0.9069
              precision    recall  f1-score   support

      normal       0.96      0.95      0.95      8225
     flagged       0.85      0.86      0.86      2640

    accuracy                           0.93     10865
   macro avg       0.90      0.91      0.91     10865
weighted avg       0.93      0.93      0.93     10865

Selected Threshold             : 0.799
Test Overall Macro F1          : 0.9054
Test Source-Balanced Macro F1  : 0.8309
Test Accuracy                  : 0.9301

Per-source Test Macro F1:
                  davidson: 0.9017
                hatexplain: 0.7591
                    jigsaw: 0.9015
      tweet_eval_offensive: 0.7614


In [21]:
save_cm(
    test_labels,
    test_preds,
    f"{LABEL} — Test (threshold={best_threshold:.3f})",
    os.path.join(
        OUT_DIR,
        "confusion_matrix.png",
    ),
)

results = {
    "model": "bertweet_lora32_four_source_v6",
    "backbone": MODEL_NAME,
    "datasets": [
        "Jigsaw Toxic Comment Classification",
        "HateXplain",
        "Davidson Hate Speech & Offensive Language",
        "TweetEval Offensive (OffensEval/OLID task)",
    ],
    "split":
        "source-aware 90/5/5; no destructive downsampling",
    "train_sampling":
        "inverse (source,label) group frequency; replacement=True",
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "lora_targets": LORA_TARGETS,
    "loss": "CrossEntropyLoss",
    "class_weights": None,
    "lr_schedule": "cosine_with_warmup",
    "warmup_ratio": WARMUP_RATIO,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "num_epochs": NUM_EPOCHS,
    "patience": PATIENCE,
    "checkpoint_selection":
        "source_balanced_validation_macro_f1",
    "threshold": round(
        float(best_threshold),
        4,
    ),
    "threshold_selection":
        "validation_source_balanced_macro_f1_fine_sweep",
    "validation_source_balanced_macro_f1":
        round(float(best_threshold_source_f1), 4),
    "validation_overall_macro_f1":
        round(float(best_threshold_overall_f1), 4),
    "test_macro_f1":
        round(float(test_f1), 4),
    "test_source_balanced_macro_f1":
        round(float(test_source_balanced_f1), 4),
    "test_accuracy":
        round(float(test_accuracy), 4),
    "test_source_macro_f1": {
        k: round(float(v), 4)
        for k, v in test_source_f1.items()
    },
    "per_class": {
        c: {
            "precision":
                round(report[c]["precision"], 4),
            "recall":
                round(report[c]["recall"], 4),
            "f1":
                round(report[c]["f1-score"], 4),
        }
        for c in list(ID2LABEL.values())
        if c in report
    },
}

with open(
    os.path.join(OUT_DIR, "results.json"),
    "w",
) as f:
    json.dump(
        results,
        f,
        indent=2,
    )

print(json.dumps(results, indent=2))


Saved confusion matrix: outputs/bertweet_lora32_four_source_v6/confusion_matrix.png
{
  "model": "bertweet_lora32_four_source_v6",
  "backbone": "vinai/bertweet-base",
  "datasets": [
    "Jigsaw Toxic Comment Classification",
    "HateXplain",
    "Davidson Hate Speech & Offensive Language",
    "TweetEval Offensive (OffensEval/OLID task)"
  ],
  "split": "source-aware 90/5/5; no destructive downsampling",
  "train_sampling": "inverse (source,label) group frequency; replacement=True",
  "lora_r": 32,
  "lora_alpha": 64,
  "lora_dropout": 0.2,
  "lora_targets": [
    "query",
    "key",
    "value",
    "dense"
  ],
  "loss": "CrossEntropyLoss",
  "class_weights": null,
  "lr_schedule": "cosine_with_warmup",
  "warmup_ratio": 0.06,
  "learning_rate": 5e-05,
  "weight_decay": 0.05,
  "num_epochs": 5,
  "patience": 2,
  "checkpoint_selection": "source_balanced_validation_macro_f1",
  "threshold": 0.799,
  "threshold_selection": "validation_source_balanced_macro_f1_fine_sweep",
  "validat

### 9.1 Source-wise and length-bias diagnostics

This section preserves the diagnostics that exposed the earlier issue: per-source performance plus the false-positive rate on **known-normal** test examples across BERTweet token-length bins.


In [22]:
analysis_df = test_df_raw.reset_index(drop=True).copy()

analysis_df["flagged_probability"] = test_probs
analysis_df["prediction"] = test_preds

analysis_df["token_length"] = [
    len(
        tokenizer.encode(
            text,
            add_special_tokens=True,
            truncation=False,
        )
    )
    for text in analysis_df["text"].tolist()
]

print("SOURCE-WISE TEST PERFORMANCE")
print("=" * 72)

source_metrics = {}

for source_name, group in analysis_df.groupby("source"):
    idx = group.index.to_numpy()

    y_true = test_labels[idx]
    y_pred = test_preds[idx]

    src_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )

    src_acc = accuracy_score(
        y_true,
        y_pred,
    )

    source_metrics[source_name] = {
        "n": int(len(group)),
        "macro_f1": float(src_f1),
        "accuracy": float(src_acc),
    }

    print(
        f"{source_name:>24}  "
        f"n={len(group):5d}  "
        f"macro_f1={src_f1:.4f}  "
        f"accuracy={src_acc:.4f}"
    )

print("\nNORMAL-ONLY FALSE-POSITIVE RATE BY TOKEN LENGTH")
print("=" * 72)

length_bins = [
    0,
    8,
    16,
    32,
    64,
    96,
    128,
    np.inf,
]

analysis_df["length_bin"] = pd.cut(
    analysis_df["token_length"],
    bins=length_bins,
    include_lowest=True,
)

normal_only = analysis_df[
    analysis_df["label"] == LABEL2ID["normal"]
].copy()

length_report = (
    normal_only.groupby(
        "length_bin",
        observed=True,
    )
    .agg(
        samples=("label", "size"),
        avg_flagged_probability=(
            "flagged_probability",
            "mean",
        ),
        false_positive_rate=(
            "prediction",
            "mean",
        ),
        avg_tokens=("token_length", "mean"),
    )
)

print(
    length_report.round(4)
)

if len(normal_only) > 1:
    corr = normal_only[
        [
            "token_length",
            "flagged_probability",
        ]
    ].corr().iloc[0, 1]
else:
    corr = float("nan")

print(
    "\nCorrelation between token length and "
    "flagged probability for NORMAL test examples: "
    f"{corr:.4f}"
)

diagnostics = {
    "source_metrics": source_metrics,
    "normal_length_flagged_probability_correlation":
        None if np.isnan(corr) else float(corr),
    "length_report": {
        str(idx): {
            key: (
                int(value)
                if key == "samples"
                else float(value)
            )
            for key, value in row.items()
        }
        for idx, row in (
            length_report
            .to_dict(orient="index")
            .items()
        )
    },
}

with open(
    os.path.join(
        OUT_DIR,
        "generalization_diagnostics.json",
    ),
    "w",
) as f:
    json.dump(
        diagnostics,
        f,
        indent=2,
    )

print(
    f"\nSaved: "
    f"{os.path.join(OUT_DIR, 'generalization_diagnostics.json')}"
)


SOURCE-WISE TEST PERFORMANCE
                davidson  n= 1239  macro_f1=0.9017  accuracy=0.9411
              hatexplain  n=  960  macro_f1=0.7591  accuracy=0.7615
                  jigsaw  n= 7965  macro_f1=0.9015  accuracy=0.9611
    tweet_eval_offensive  n=  701  macro_f1=0.7614  accuracy=0.7903

NORMAL-ONLY FALSE-POSITIVE RATE BY TOKEN LENGTH
               samples  avg_flagged_probability  false_positive_rate  \
length_bin                                                             
(-0.001, 8.0]      213                   0.0950               0.0657   
(8.0, 16.0]       1022                   0.0953               0.0734   
(16.0, 32.0]      1803                   0.0851               0.0544   
(32.0, 64.0]      2215                   0.0782               0.0546   
(64.0, 96.0]      1032                   0.0563               0.0320   
(96.0, 128.0]      605                   0.0641               0.0264   
(128.0, inf]      1335                   0.0661               0.0262   

 

## 10. Clean deployable export

v6 permanently incorporates the tokenizer lesson from v5:

- merge the LoRA adapter into the base model;
- start from a clean artifact directory;
- save the merged model;
- **do not call `tokenizer.save_pretrained()` for BERTweet**;
- copy the original `vocab.txt` and `bpe.codes` from `vinai/bertweet-base`;
- load the final tokenizer explicitly with `BertweetTokenizer`;
- assert that final-artifact token IDs match the training tokenizer before any regression test or ZIP is created.


In [23]:
# Reload the best adapter and merge it into a clean base model.
base_for_merge = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

peft_model = PeftModel.from_pretrained(
    base_for_merge,
    best_ckpt,
)

merged = peft_model.merge_and_unload()

merged.config.id2label = {
    int(k): v
    for k, v in ID2LABEL.items()
}

merged.config.label2id = LABEL2ID

# Always build the deployable artifact from a clean directory.
if os.path.exists(ARTIFACT_DIR):
    shutil.rmtree(ARTIFACT_DIR)

os.makedirs(
    ARTIFACT_DIR,
    exist_ok=True,
)

merged.save_pretrained(
    ARTIFACT_DIR,
    safe_serialization=True,
)

# IMPORTANT:
# Do NOT call tokenizer.save_pretrained() here.
# Copy the original BERTweet vocabulary + BPE codes directly.
for filename in ["vocab.txt", "bpe.codes"]:
    src_path = hf_hub_download(
        repo_id=MODEL_NAME,
        filename=filename,
    )

    shutil.copy2(
        src_path,
        os.path.join(
            ARTIFACT_DIR,
            filename,
        ),
    )

    print(f"Copied tokenizer file: {filename}")

# Add explicit tokenizer metadata for portability.
tokenizer_config = {
    "tokenizer_class": "BertweetTokenizer",
    "normalization": True,
    "model_max_length": MAX_LENGTH,
    "bos_token": "<s>",
    "eos_token": "</s>",
    "sep_token": "</s>",
    "cls_token": "<s>",
    "unk_token": "<unk>",
    "pad_token": "<pad>",
    "mask_token": "<mask>",
}

with open(
    os.path.join(
        ARTIFACT_DIR,
        "tokenizer_config.json",
    ),
    "w",
) as f:
    json.dump(
        tokenizer_config,
        f,
        indent=2,
    )

special_tokens = {
    "bos_token": "<s>",
    "eos_token": "</s>",
    "sep_token": "</s>",
    "cls_token": "<s>",
    "unk_token": "<unk>",
    "pad_token": "<pad>",
    "mask_token": "<mask>",
}

with open(
    os.path.join(
        ARTIFACT_DIR,
        "special_tokens_map.json",
    ),
    "w",
) as f:
    json.dump(
        special_tokens,
        f,
        indent=2,
    )

print(
    f"Merged model + original BERTweet tokenizer files "
    f"written to {ARTIFACT_DIR}"
)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Copied tokenizer file: vocab.txt
Copied tokenizer file: bpe.codes
Merged model + original BERTweet tokenizer files written to artifacts/openstream-moderation-v6


In [24]:
# Preserve the LoRA adapter for future continued training.
adapter_dir = os.path.join(
    ARTIFACT_DIR,
    "lora_adapter",
)

shutil.copytree(
    best_ckpt,
    adapter_dir,
)

metadata = {
    "name": "openstream-moderation",
    "version": "v6",
    "task": "sequence-classification",
    "num_labels": NUM_LABELS,
    "id2label": ID2LABEL,
    "label2id": LABEL2ID,
    "max_length": MAX_LENGTH,
    "backbone": MODEL_NAME,
    "classification_threshold":
        float(best_threshold),
    "threshold_selection":
        "validation_source_balanced_macro_f1_fine_sweep",
    "checkpoint_selection":
        "source_balanced_validation_macro_f1",
    "datasets": [
        "Jigsaw Toxic Comment Classification",
        "HateXplain",
        "Davidson Hate Speech & Offensive Language",
        "TweetEval Offensive / OffensEval / OLID",
    ],
    "split":
        "source-aware 90/5/5",
    "train_sampling":
        "inverse source/class group frequency; all rows retained in pool",
    "training": {
        "method": "LoRA (merged)",
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "lora_targets": LORA_TARGETS,
        "loss": "CrossEntropyLoss",
        "class_weights": None,
        "lr_schedule": "cosine with warmup",
        "warmup_ratio": WARMUP_RATIO,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "max_epochs": NUM_EPOCHS,
        "early_stopping_patience": PATIENCE,
    },
    "metrics": results,
    "preprocessing":
        "minimal whitespace cleanup; BertweetTokenizer normalization=True",
    "tokenizer_export":
        "original vinai/bertweet-base vocab.txt + bpe.codes",
}

with open(
    os.path.join(
        ARTIFACT_DIR,
        "metadata.json",
    ),
    "w",
) as f:
    json.dump(
        metadata,
        f,
        indent=2,
    )

# Preserve the existing serving threshold.json contract.
with open(
    os.path.join(
        ARTIFACT_DIR,
        "threshold.json",
    ),
    "w",
) as f:
    json.dump(
        {
            "threshold":
                float(best_threshold),
            "validation_macro_f1":
                float(best_threshold_source_f1),
            "selection":
                "validation source-balanced Macro F1 sweep "
                "from 0.20 to 0.80 in 0.001 increments",
        },
        f,
        indent=2,
    )

total = sum(
    os.path.getsize(
        os.path.join(dp, filename)
    )
    for dp, _, filenames
    in os.walk(ARTIFACT_DIR)
    for filename in filenames
)

print(f"Artifact directory: {ARTIFACT_DIR}")
print(f"Total size: {total / 1e6:.1f} MB")

for filename in sorted(
    os.listdir(ARTIFACT_DIR)
):
    print(f"  {filename}")


Artifact directory: artifacts/openstream-moderation-v6
Total size: 565.2 MB
  bpe.codes
  config.json
  lora_adapter
  metadata.json
  model.safetensors
  special_tokens_map.json
  threshold.json
  tokenizer_config.json
  vocab.txt


## 11. Verify the exact disk artifact, then run the 15-case regression suite

The tokenizer check is a hard gate. If the copied artifact tokenizer produces different IDs or masks from the training tokenizer, the notebook raises an error **before packaging**.

The `classify()` response dictionary below is intentionally unchanged from v5.


In [25]:
# Load the exact final artifact from disk.
verify_tok = BertweetTokenizer(
    vocab_file=os.path.join(
        ARTIFACT_DIR,
        "vocab.txt",
    ),
    merges_file=os.path.join(
        ARTIFACT_DIR,
        "bpe.codes",
    ),
    normalization=True,
)

verify_model = AutoModelForSequenceClassification.from_pretrained(
    ARTIFACT_DIR
).to(DEVICE)

verify_model.eval()

with open(
    os.path.join(
        ARTIFACT_DIR,
        "threshold.json",
    ),
    "r",
) as f:
    verify_threshold = float(
        json.load(f)["threshold"]
    )


def clean_tweet(text: str) -> str:
    if not isinstance(text, str):
        return ""

    return re.sub(
        r"\s+",
        " ",
        text,
    ).strip()


# Hard tokenizer consistency gate using representative short/long text.
TOKENIZER_CHECK_SAMPLES = [
    "hello I am anutej",
    "Cold starts are a tax you pay for free hosting. Worth it for a side project, unacceptable anywhere else.",
    "you are an idiot",
    "I want to kill the person",
    "You are a useless idiot who contributes nothing, and I honestly cannot understand why anyone still listens to anything you say.",
]

print("FINAL TOKENIZER CONSISTENCY CHECK")
print("=" * 90)

for text in TOKENIZER_CHECK_SAMPLES:
    original_enc = tokenizer(
        clean_tweet(text),
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )

    artifact_enc = verify_tok(
        clean_tweet(text),
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )

    same_ids = torch.equal(
        original_enc["input_ids"],
        artifact_enc["input_ids"],
    )

    same_mask = torch.equal(
        original_enc["attention_mask"],
        artifact_enc["attention_mask"],
    )

    print(text)
    print(f"  same input_ids     : {same_ids}")
    print(f"  same attention_mask: {same_mask}")

    if not (same_ids and same_mask):
        raise RuntimeError(
            "Final BERTweet tokenizer does not match the "
            "training tokenizer. Packaging stopped."
        )


def classify(text: str) -> dict:
    cleaned = clean_tweet(text)

    enc = verify_tok(
        cleaned,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    ).to(DEVICE)

    with torch.no_grad():
        logits = verify_model(
            input_ids=enc["input_ids"],
            attention_mask=enc["attention_mask"],
        ).logits

        probs = F.softmax(
            logits,
            dim=-1,
        ).squeeze(0).cpu().numpy()

    flagged_prob = float(probs[1])
    pred = int(
        flagged_prob >= verify_threshold
    )

    threshold_margin = abs(
        flagged_prob - verify_threshold
    )

    # IMPORTANT:
    # Keep this response structure unchanged for the existing applications.
    return {
        "label": ID2LABEL[pred],
        "flagged_probability": round(
            flagged_prob,
            4,
        ),
        "threshold": round(
            verify_threshold,
            4,
        ),
        "threshold_margin": round(
            float(threshold_margin),
            4,
        ),
        "margin_direction":
            "above"
            if flagged_prob >= verify_threshold
            else "below",
        "scores": {
            ID2LABEL[i]: round(
                float(probs[i]),
                4,
            )
            for i in range(NUM_LABELS)
        },
    }


SANITY_SAMPLES = [
    # 10 longer statements: 6 normal + 4 flagged.
    {
        "expected": "normal",
        "text": "Cold starts are a tax you pay for free hosting. Worth it for a side project, unacceptable anywhere else."
    },
    {
        "expected": "normal",
        "text": "Tried explaining JOIN FETCH to someone and realised halfway through that I understood it better than I thought."
    },
    {
        "expected": "normal",
        "text": "Learned the difference between authentication and authorisation properly today, roughly a decade later than I should have."
    },
    {
        "expected": "normal",
        "text": "Spent most of the afternoon debugging a Docker networking issue, and after changing three different settings I discovered the real problem was a single incorrect port in my configuration."
    },
    {
        "expected": "normal",
        "text": "I went to the university library after class to finish my assignment, stayed there for a few hours, and finally got the API integration working before heading home."
    },
    {
        "expected": "normal",
        "text": "I am going to attend the protest against the governor this weekend because I want to hear what the speakers have to say and understand the issue better."
    },
    {
        "expected": "flagged",
        "text": "You keep ruining every task you touch and everyone on this team would be better off if you stopped pretending you knew what you were doing."
    },
    {
        "expected": "flagged",
        "text": "If that person comes near me again I am going to hurt them badly and make sure they remember not to mess with me."
    },
    {
        "expected": "flagged",
        "text": "You are a useless idiot who contributes nothing, and I honestly cannot understand why anyone still listens to anything you say."
    },
    {
        "expected": "flagged",
        "text": "I have had enough of that guy constantly bothering me, and the next time he does it I am going to make him regret ever speaking to me."
    },

    # 5 short statements: 3 normal + 2 flagged.
    {
        "expected": "normal",
        "text": "hello I am anutej"
    },
    {
        "expected": "normal",
        "text": "I dont like real madriad"
    },
    {
        "expected": "normal",
        "text": "great work today"
    },
    {
        "expected": "flagged",
        "text": "you are an idiot"
    },
    {
        "expected": "flagged",
        "text": "I want to kill the person"
    },
]

print("\nPRE-DOWNLOAD 15-SAMPLE REGRESSION SUITE")
print("=" * 90)

passed = 0
suite_rows = []

for i, sample in enumerate(
    SANITY_SAMPLES,
    start=1,
):
    result = classify(
        sample["text"]
    )

    predicted = result["label"]
    expected = sample["expected"]

    ok = predicted == expected
    passed += int(ok)

    suite_rows.append({
        "index": i,
        "expected": expected,
        "predicted": predicted,
        "flagged_probability":
            result["flagged_probability"],
        "threshold":
            result["threshold"],
        "passed": bool(ok),
        "text": sample["text"],
    })

    print(
        f"{i:02d}. "
        f"{'PASS' if ok else 'FAIL'}  "
        f"expected={expected:<7} "
        f"predicted={predicted:<7}  "
        f"flagged={result['flagged_probability']:.4f}  "
        f"threshold={result['threshold']:.4f}"
    )

    print(
        f"    {sample['text']}"
    )

print(
    f"\nSanity suite score: "
    f"{passed}/{len(SANITY_SAMPLES)}"
)

with open(
    os.path.join(
        OUT_DIR,
        "sanity_suite_results.json",
    ),
    "w",
) as f:
    json.dump(
        suite_rows,
        f,
        indent=2,
    )

print(
    f"Saved: "
    f"{os.path.join(OUT_DIR, 'sanity_suite_results.json')}"
)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

FINAL TOKENIZER CONSISTENCY CHECK
hello I am anutej
  same input_ids     : True
  same attention_mask: True
Cold starts are a tax you pay for free hosting. Worth it for a side project, unacceptable anywhere else.
  same input_ids     : True
  same attention_mask: True
you are an idiot
  same input_ids     : True
  same attention_mask: True
I want to kill the person
  same input_ids     : True
  same attention_mask: True
You are a useless idiot who contributes nothing, and I honestly cannot understand why anyone still listens to anything you say.
  same input_ids     : True
  same attention_mask: True

PRE-DOWNLOAD 15-SAMPLE REGRESSION SUITE
01. PASS  expected=normal  predicted=normal   flagged=0.0021  threshold=0.7990
    Cold starts are a tax you pay for free hosting. Worth it for a side project, unacceptable anywhere else.
02. PASS  expected=normal  predicted=normal   flagged=0.0068  threshold=0.7990
    Tried explaining JOIN FETCH to someone and realised halfway through that I under

### 11.5 Interactive Testing Environment

Use this only after the artifact tokenizer consistency gate and 15-sample suite have completed.


In [26]:
import ipywidgets as widgets
from IPython.display import display, clear_output

text_input = widgets.Textarea(
    value='',
    placeholder='Type a message to classify...',
    description='Input:',
    layout=widgets.Layout(width='80%', height='80px')
)

analyze_button = widgets.Button(
    description='Classify Text',
    button_style='primary',
    icon='search'
)

output_area = widgets.Output()


def on_analyze_clicked(b):
    with output_area:
        clear_output()
        text = text_input.value.strip()
        if not text:
            print("Please enter some text to classify.")
            return

        print(f"Analyzing: '{text}'\n")
        try:
            result = classify(text)
            label = result["label"].upper()
            flagged_prob = result["flagged_probability"]
            threshold = result["threshold"]
            margin = result["threshold_margin"]
            direction = result["margin_direction"]

            print(f"Prediction: {label}")
            print(f"Flagged probability : {flagged_prob:.4f}")
            print(f"Decision threshold  : {threshold:.4f}")
            print(f"Threshold margin    : {margin:.4f} {direction} threshold\n")
            print("All Class Scores:")
            for cls, score in result["scores"].items():
                print(f"  - {cls.ljust(10)}: {score:.4f}")
        except Exception as e:
            print(f"Error during classification: {e}")


analyze_button.on_click(on_analyze_clicked)
display(widgets.VBox([text_input, analyze_button, output_area]))


## 12. Package for download

Packaging happens only after the final disk artifact has passed the tokenizer consistency gate and the 15-case regression suite.


In [27]:
shutil.make_archive(
    ARCHIVE_NAME,
    "zip",
    ARTIFACT_DIR,
)

size = os.path.getsize(
    f"{ARCHIVE_NAME}.zip"
) / 1e6

print(
    f"{ARCHIVE_NAME}.zip  "
    f"({size:.1f} MB)"
)


openstream-moderation-model-v6.zip  (457.4 MB)


In [28]:
# Colab only
try:
    from google.colab import files
    files.download(f"{ARCHIVE_NAME}.zip")
except ImportError:
    print("Not running in Colab — download the zip from the file browser.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---

## Notes for the serving layer

**Response contract:** unchanged from v5. Labels remain `normal=0`, `flagged=1`, and the backend must compare `flagged_probability` against the value in `threshold.json`.

**Tokenizer:** use BERTweet explicitly with `normalization=True`. The v6 artifact packages the original `vinai/bertweet-base` `vocab.txt` and `bpe.codes`, because re-saving the tokenizer previously changed tokenization.

Recommended backend loading:

```python
from transformers import BertweetTokenizer, AutoModelForSequenceClassification

tokenizer = BertweetTokenizer.from_pretrained(
    MODEL_ID,
    normalization=True,
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
)
```

The API response structure does not need to change.
